# ResNet-18 — full re-verification, GaussianNoise(0.05) removed (all tests in one notebook)

**Why this notebook exists.** Supervisor review flagged the `GaussianNoise(0.05)` input-noise
layer at the start of this study's ResNet-18 architecture and asked for a rerun **without** it,
across **every test in the manuscript where ResNet-18 is reported**, so the whole set of numbers
can be refreshed together before re-running the McNemar/significance tests and editing the paper.

**Nothing else about the architecture changes.** The graduated `StochasticDepth` residual blocks
(survival rates 0.95 -> 0.72 across the eight blocks), `SpatialDropout1D(dr=0.1)` inside each
residual block, the `Adam(3e-4)` optimiser, label-smoothed (`0.1`) categorical cross-entropy, the
`StandardScaler` fit on the training fold only, and the `EarlyStopping` / `ReduceLROnPlateau`
schedule are all byte-identical to the verified baseline architecture that produced this paper's
Table 3, Table 4, Table 6, Table 7, and the supplementary ambiguity/robustness tables. The **only**
line that changes, in `build_resnet18_no_noise`, is:

```python
# before (verified baseline, in the manuscript today):
inp = Input((L,1)); x = GaussianNoise(0.05)(inp)
# after (this notebook):
inp = Input((L,1)); x = inp
```

**Scope — every ResNet-18 result in the paper, in one run:**

| # | Test | Matches | Grid |
|---|------|---------|------|
| A | Fragment-length sweep | Table 4 | 7 lengths (300,400,500,750,1000,1500,full) x 3 seeds |
| B | Nine-seed robustness | Supp. Table S11 | 300/400/500 bp x 9 seeds (reuses A's seeds 0-2, adds 3-8) |
| C | Encoding comparison | Table 7 / Supp. S5 | 4 encodings (raw 6-mer, TF-IDF 6-mer, Word2Vec 6-mer, Word2Vec-TF-IDF) x 3 seeds, full length |
| D | PR+RT amplicon | Table 6 | PR only, RT only, PR+RT, PR+RT+200bp-dropout x 3 seeds |
| E | Ambiguity injection | Table 7 (ambiguity) / old Table 12b | 3 lengths (300,500,full) x 3 rates (0%,5%,10%) x 3 seeds |

Only **ResNet-18** is retrained in every section above — the linear SVM, random forest and VGG-16
numbers already in the paper are unaffected by this change and are not rerun for their own sake.
The one exception: in sections A and D, this notebook **also** fits a plain linear SVM (TF-IDF
6-mer, `LinearSVC`, same settings as the main study) on the *same* split/seed as each ResNet-18
run, and saves **both models' per-sequence predictions**. That costs almost nothing (SVM fits in
seconds, CPU only) and is what lets you redo the SVM-vs-ResNet-18 McNemar tests (Table 5, Table
S9/S13) once this finishes, with correctly matched predictions.

**GPU needed.** Yes, for every ResNet-18 fit. **Runtime.** This study's own Supplementary Table
S17 measured this exact architecture at **32.8 s/epoch on a T4**. With `EPOCHS=120` and
`EarlyStopping(patience=15)`, a single fit costs anywhere from roughly 10 minutes (if early
stopping trips early) up to about 65 minutes (if it runs the full 120 epochs) -- so across all 90
ResNet-18 fits (A=21, B=18, C=12, D=12, E=27), a realistic total is **~45 GPU-hours**, and a
worst-case total (every fit runs all 120 epochs) is **~98 GPU-hours**. Either way **this will not
fit in one 12-hour Kaggle session** -- plan for 4-9 separate sessions. This is exactly what
Section 2b (just below) is for: it lets each new session pick up exactly where the last one
stopped, instead of restarting from scratch. A sensible order: run **C and D first**
(12 fits each, cheapest, finish in ~6-13 h realistic/worst case -- fits in one session), then
**A and B**, and do **E last**, splitting it by ambiguity rate across two sessions if needed.

**Dataset.** Reads the same files you have already used for every other notebook in this project:

- `/kaggle/input/datasets/yashrajtheking/msc-work-final/prepared_sequences_clean.csv`
  (columns `sequence`, `subtype`, `cluster` — the CD-HIT-97% cluster id used for the
  leakage-controlled, grouped split)
- `/kaggle/input/datasets/yashrajtheking/msc-work-final/t11_hxb2_coords.csv`
  (the PR/RT HXB2 coordinates already extracted for the amplicon tests — reused as-is, so this
  notebook does **not** repeat the slow reference-alignment step; **edit `COORDS_PATH` below** if
  your copy lives at a different path)

Attach both the same way you always do (Add Data -> Your Datasets). **Note:** this notebook computes
its own CD-HIT-97% clustering (Section 3b) from the sequences in the first file — it does **not**
reuse that file's `cluster` column, which is a coarser MinHash clustering, not CD-HIT-97%. See
Section 3b for why that distinction matters.

**Outputs** (all in `/kaggle/working/`, one pair of CSVs per section, plus a combined zip at the
end): `resnet18_A_fragmentation_raw.csv` / `_predictions.csv` / `_summary.csv`,
`resnet18_B_nineseed_raw.csv` / `_summary.csv`, `resnet18_C_encoding_raw.csv` / `_summary.csv`,
`resnet18_D_amplicon_raw.csv` / `_predictions.csv` / `_summary.csv`,
`resnet18_E_ambiguity_raw.csv` / `_summary.csv`.

### 1. Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, random
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score, accuracy_score
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, Add,
    GlobalAveragePooling1D, Dropout, Dense, MaxPooling1D, SpatialDropout1D, Layer)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from gensim.models import Word2Vec
print("TF", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

### 2. Config

In [ ]:
DATA_PATH   = "/kaggle/input/datasets/yashrajtheking/msc-work-final/prepared_sequences_clean.csv"
COORDS_PATH = "/kaggle/input/datasets/yashrajtheking/msc-work-final/t11_hxb2_coords.csv"  # <-- EDIT if your path differs

SEEDS            = [0, 1, 2]                                 # main 3 seeds, used throughout
EXTRA_SEEDS      = [3, 4, 5, 6, 7, 8]                         # for the 9-seed robustness check (Section B)
LENGTHS_FULL     = [300, 400, 500, 750, 1000, 1500, None]     # Section A (fragmentation sweep); None = full length
LENGTHS_ROBUST   = [300, 400, 500]                            # Section B (nine-seed robustness)
LENGTHS_AMBIG    = [300, 500, None]                           # Section E (ambiguity)
AMBIG_RATES      = [0.0, 0.05, 0.10]                          # Section E
VEC_SIZE         = 100                                        # Word2Vec embedding size (matches the rest of the study)
EPOCHS           = 120
BATCH            = 48
OUT              = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

### 2b. Resume across Kaggle sessions (read this before your first run)

Every section below already skips any `(length, seed)` / `(encoding, seed)` / etc. combination
it finds in its own CSV, so re-running a section never repeats finished work -- **as long as
that CSV is sitting in `/kaggle/working/` when the section runs.** The catch: Kaggle wipes
`/kaggle/working/` clean at the start of every *new* session, so without this cell, a fresh
session has nothing to resume from and silently starts over.

**How to actually carry progress forward, session to session:**

1. Before a session ends (whether it finishes a section or Kaggle cuts it off at the time
   limit), go to **Save Version -> Quick Save** (not "Save & Run All" -- Quick Save snapshots
   whatever is in `/kaggle/working/` right now, without re-running anything, so it also works
   mid-run if you're about to run out of time).
2. Start the next session from this same notebook. Click **Add Data -> Your Work -> Notebook
   Output**, pick this notebook, and select the version you just saved. Kaggle mounts its files
   under `/kaggle/input/<this-notebook-slug>/`.
3. Set `RESUME_INPUT_DIR` in the cell below to that mounted path, then **Run All**. This cell
   copies every `*_raw.csv` / `*_predictions.csv` it finds there into `/kaggle/working/` *before*
   any section's resume-check runs -- so each section sees exactly what was already finished and
   only computes what's left.

Leave `RESUME_INPUT_DIR = ""` for the very first run (nothing to resume from yet), and for any
run that's still inside the *same* live session as a previous cell execution (files are already
sitting in `/kaggle/working/` in that case -- no copy needed).

**To run it in deliberate chunks instead of racing the clock:** comment out the sections you
don't want this session (e.g. wrap Section E's two code cells in `if False:` temporarily), Quick
Save at the end, then come back for the rest with `RESUME_INPUT_DIR` pointed at that save.

In [ ]:
import glob, shutil

RESUME_INPUT_DIR = ""   # <-- EDIT: e.g. "/kaggle/input/resnet18-full-reverification-no-gaussian-noise"

RESUMABLE_FILES = [
    "pol_resnet18_full_cdhit97.clstr",   # the CD-HIT-97% clustering itself -- skip ~1-2 min of reclustering
    "resnet18_A_fragmentation_raw.csv", "resnet18_A_fragmentation_predictions.csv",
    "resnet18_B_nineseed_raw.csv",
    "resnet18_C_encoding_raw.csv",
    "resnet18_D_amplicon_raw.csv", "resnet18_D_amplicon_predictions.csv",
    "resnet18_E_ambiguity_raw.csv",
]

if RESUME_INPUT_DIR:
    restored = 0
    for fname in RESUMABLE_FILES:
        src = os.path.join(RESUME_INPUT_DIR, fname)
        dst = os.path.join(OUT, fname)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            restored += 1
            print(f"[resume] restored {fname} ({os.path.getsize(src):,} bytes) from previous session")
    if restored == 0:
        print("[resume] RESUME_INPUT_DIR is set but no matching files were found there -- "
              "double-check the mounted path (ls it below if unsure), or this really is a fresh start.")
else:
    print("[resume] RESUME_INPUT_DIR is empty -- starting fresh, or continuing inside the same live session.")


### 2c. Progress check (optional) -- see what's already done before you run anything

Safe to run any time, including right after the resume cell above and before Section A. Shows
`done / total` combinations per section from whatever CSVs currently exist in `/kaggle/working/`.
One caveat: Section B's count only reflects its own saved file -- if Section A already finished
the 300/400/500 bp lengths, Section B will reuse those seeds automatically the moment it runs,
even before `resnet18_B_nineseed_raw.csv` exists, so this check can under-report B right after a
fresh A-only resume.

In [ ]:
def _combo_progress(raw_path, key_cols, total):
    if not os.path.exists(raw_path):
        return 0, total
    done = len(pd.read_csv(raw_path)[key_cols].drop_duplicates())
    return done, total

print("Section A (fragmentation):    %d / %d" % _combo_progress(
    f"{OUT}/resnet18_A_fragmentation_raw.csv", ["length","seed"], len(LENGTHS_FULL)*len(SEEDS)))
print("Section B (nine-seed robust.):%d / %d" % _combo_progress(
    f"{OUT}/resnet18_B_nineseed_raw.csv", ["length","seed"], len(LENGTHS_ROBUST)*(len(SEEDS)+len(EXTRA_SEEDS))))
print("Section C (encoding):         %d / %d" % _combo_progress(
    f"{OUT}/resnet18_C_encoding_raw.csv", ["encoding","seed"], 4*len(SEEDS)))
print("Section D (PR/RT amplicon):   %d / %d" % _combo_progress(
    f"{OUT}/resnet18_D_amplicon_raw.csv", ["input","seed"], 4*len(SEEDS)))
print("Section E (ambiguity):        %d / %d" % _combo_progress(
    f"{OUT}/resnet18_E_ambiguity_raw.csv", ["length","ambiguity_rate","seed"],
    len(LENGTHS_AMBIG)*len(AMBIG_RATES)*len(SEEDS)))


### 3. Load data

In [ ]:
df = pd.read_csv(DATA_PATH)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
print(f"{len(df)} seqs | {n_cls} classes: {CLASSES}")

coords_df = pd.read_csv(COORDS_PATH)
good = coords_df.dropna(subset=["pr", "rt"]).reset_index(drop=True)
print(f"{len(good)} sequences with both PR and RT extracted (for Section D)")

### 3b. CD-HIT-97% clustering (the real leakage-controlled grouping variable)

`prepared_sequences_clean.csv`'s own `cluster` column is the **MinHash** cluster id -- a cheap,
coarse pre-filter used earlier in this study, **not** the CD-HIT-97%-identity clustering that
every reported split in the paper (Table 3, Table 4, Table 6, Table 7, and every supplementary
significance/robustness table) actually groups train/val/test on. Using the MinHash column here
would silently give this rerun a *different, weaker* leakage control than the rest of the study:
a near-duplicate sequence can share a MinHash cluster yet still fail the finer 97%-identity test,
so it could end up split across train and test, inflating every number in this notebook relative
to the real baseline it is supposed to match.

This cell runs the exact `cd-hit-est` command used throughout the rest of this project
(`T16`/`T22`/`T23`: `-c 0.97 -n 10`) directly on these 24,072 sequences, so `grp_all` below is the
real CD-HIT-97% clustering -- the same ~7,182 clusters reported in Table 1 -- not a stand-in.

In [ ]:
import subprocess, shutil

if shutil.which("cd-hit-est") is None:
    print("cd-hit-est not found, installing...")
    subprocess.run("conda install -y -c bioconda cd-hit 2>/dev/null || true", shell=True)
    if shutil.which("cd-hit-est") is None:
        subprocess.run(
            "wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz "
            "&& tar xzf cd-hit-v4.8.1-2019-0228.tar.gz && cd cd-hit-v4.8.1-2019-0228 && make -j4 "
            "&& cp cd-hit-est /usr/local/bin/ && cp cd-hit /usr/local/bin/", shell=True)
print("cd-hit-est available:", shutil.which("cd-hit-est") is not None)

CDHIT_FASTA = f"{OUT}/pol_resnet18_full.fasta"
CDHIT_OUT   = f"{OUT}/pol_resnet18_full_cdhit97"

def parse_cdhit_clusters(clstr_path, n_seqs):
    labels = np.full(n_seqs, -1, dtype=int)
    cid = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                cid += 1
            else:
                idx = int(line.split(">")[1].split("...")[0])
                labels[idx] = cid
    assert (labels >= 0).all(), "some sequences never got a cluster id -- CD-HIT output does not match the input FASTA"
    return labels

# resume-safe: if a previous session already produced the .clstr file (brought forward via
# RESUME_INPUT_DIR in Section 2b), reuse it instead of reclustering from scratch
if os.path.exists(CDHIT_OUT + ".clstr"):
    grp_all = parse_cdhit_clusters(CDHIT_OUT + ".clstr", len(df))
    print(f"[resume] reused existing CD-HIT-97% clustering: {len(set(grp_all))} clusters.")
else:
    with open(CDHIT_FASTA, "w") as f:
        for i, s in enumerate(X_all):
            f.write(f">{i}\n{s}\n")
    cmd = f"cd-hit-est -i {CDHIT_FASTA} -o {CDHIT_OUT} -c 0.97 -n 10 -M 16000 -T 4 -d 0"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    assert r.returncode == 0 and os.path.exists(CDHIT_OUT + ".clstr"), \
        f"CD-HIT failed (returncode={r.returncode}):\n{r.stderr[-2000:]}"
    grp_all = parse_cdhit_clusters(CDHIT_OUT + ".clstr", len(df))

print(f"CD-HIT 97% identity: {len(set(grp_all))} clusters, "
      f"largest = {pd.Series(grp_all).value_counts().iloc[0]} sequences "
      f"(this study's Table 1 reports 7,182 CD-HIT-97% clusters over the full 24,072-sequence dataset)")


### 4. Shared helpers: split, fragmentation, encoders, ambiguity, dropout, metrics

In [ ]:
# --- leakage-free split (depends on seed only) ---
def grouped_split(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te

# --- fragmentation: one random contiguous window of length L per sequence ---
def fragment(seqs, L, rng):
    if L is None: return list(seqs)
    out = []
    for s in seqs:
        if len(s) <= L: out.append(s)
        else:
            st = int(rng.integers(0, len(s) - L + 1))
            out.append(s[st:st+L])
    return out

def kmers(seq, k=6): return [seq[i:i+k] for i in range(len(seq)-k+1)]

def make_tfidf(train, k=6, max_features=512):
    vec = TfidfVectorizer(max_features=max_features)
    vec.fit([" ".join(kmers(s,k)) for s in train])
    return lambda seqs: vec.transform([" ".join(kmers(s,k)) for s in seqs]).toarray().astype(np.float32)

def make_kfreq(train, k=6):
    # "raw 6-mer" encoding used for VGG-16 throughout this study: per-sequence k-mer
    # frequency vector over the vocabulary observed in the training fold
    vocab = sorted({km for s in train for km in kmers(s,k)}); idx = {kk:i for i,kk in enumerate(vocab)}
    def enc(seqs):
        out = np.zeros((len(seqs), len(vocab)), np.float32)
        for i, s in enumerate(seqs):
            for km in kmers(s,k):
                if km in idx: out[i, idx[km]] += 1
            t = out[i].sum()
            if t: out[i] /= t
        return out
    return enc

def make_w2v(train, k=6, vs=VEC_SIZE):
    mdl = Word2Vec([kmers(s, k) for s in train], vector_size=vs, window=5,
                   min_count=1, workers=4, seed=42, epochs=10)
    def enc(seqs):
        out = []
        for s in seqs:
            v = [mdl.wv[w] for w in kmers(s, k) if w in mdl.wv]
            out.append(np.mean(v, 0) if v else np.zeros(vs))
        return np.array(out, np.float32)
    return enc

def make_w2v_tfidf_hybrid(train, k=6, vs=VEC_SIZE, max_features=512):
    # Word2Vec-TF-IDF hybrid: concatenates the TF-IDF vector and the averaged Word2Vec vector
    tfidf_enc = make_tfidf(train, k, max_features)
    w2v_enc = make_w2v(train, k, vs)
    return lambda seqs: np.concatenate([tfidf_enc(seqs), w2v_enc(seqs)], axis=1)

# --- ambiguity injection (IUPAC codes; simulates a mixed quasispecies position) ---
AMBIG = list("RYKMSWBDHVN")
def inject_ambiguity(seq, rate, rng):
    if rate <= 0: return seq
    s = list(seq)
    n_to_change = int(len(s) * rate)
    if n_to_change == 0: return seq
    idx = rng.choice(len(s), size=n_to_change, replace=False)
    for i in idx: s[i] = rng.choice(AMBIG)
    return "".join(s)

# --- PR+RT 200bp dropout (simulated partial amplification failure) ---
def dropout_200nt(seq, rng):
    if len(seq) <= 220: return seq
    start = rng.integers(50, len(seq) - 200 - 50)
    return seq[:start] + seq[start+200:]

def score(y_true, y_pred):
    return dict(f1=round(f1_score(y_true, y_pred, average="macro"),4),
                mcc=round(matthews_corrcoef(y_true, y_pred),4),
                bal_acc=round(balanced_accuracy_score(y_true, y_pred),4),
                acc=round(accuracy_score(y_true, y_pred),4))

### 5. ResNet-18 (1D), GaussianNoise removed

Identical to the main study's `build_resnet18`, with the input no longer passed through
`GaussianNoise(0.05)` before the first convolution. Everything else — `StochasticDepth`, the
graduated per-block survival rates, `SpatialDropout1D(dr=0.1)`, `Adam(3e-4)`, label-smoothed
cross-entropy, the training loop and callbacks — is unchanged.

In [ ]:
class StochasticDepth(Layer):
    def __init__(self, p=0.85, **kw): super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training: return x
        r = tf.random.uniform([tf.shape(x)[0],1,1]); return x*tf.floor(self.p+r)/self.p
    def get_config(self): return {**super().get_config(), "p": self.p}

def res_block(x, f, stride=1, sp=0.85, dr=0.1):
    sc = x
    x = Conv1D(f,3,strides=stride,padding="same",use_bias=False)(x)
    x = ReLU()(BatchNormalization()(x)); x = SpatialDropout1D(dr)(x)
    x = BatchNormalization()(Conv1D(f,3,padding="same",use_bias=False)(x))
    if stride!=1 or sc.shape[-1]!=f:
        sc = BatchNormalization()(Conv1D(f,1,strides=stride,padding="same",use_bias=False)(sc))
    return ReLU()(Add()([StochasticDepth(sp)(x), sc]))

def build_resnet18_no_noise(L, n):
    inp = Input((L,1))
    x = inp                                              # <-- GaussianNoise(0.05) removed; the only change
    x = ReLU()(BatchNormalization()(Conv1D(32,7,strides=2,padding="same",use_bias=False)(x)))
    x = MaxPooling1D(3,strides=2,padding="same")(x)
    for ff,sp in [(32,.95),(32,.92)]:   x = res_block(x,ff,1,sp)
    for ff,sp in [(64,.88),(64,.85)]:   x = res_block(x,ff,2 if ff!=x.shape[-1] else 1,sp)
    for ff,sp in [(128,.80),(128,.78)]: x = res_block(x,ff,2 if ff!=x.shape[-1] else 1,sp)
    for ff,sp in [(256,.75),(256,.72)]: x = res_block(x,ff,2 if ff!=x.shape[-1] else 1,sp)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    m = Model(inp, Dense(n,activation="softmax")(x))
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def train_resnet(ftr, fva, fte, ytr, yva, cw):
    sc = StandardScaler()                                 # fit on the training fold only
    a = sc.fit_transform(ftr)[...,None]; b = sc.transform(fva)[...,None]; c = sc.transform(fte)[...,None]
    tf.keras.backend.clear_session()
    m = build_resnet18_no_noise(a.shape[1], n_cls)
    m.fit(a, to_categorical(ytr,n_cls), validation_data=(b,to_categorical(yva,n_cls)),
          epochs=EPOCHS, batch_size=BATCH, class_weight=cw, shuffle=True, verbose=0,
          callbacks=[EarlyStopping("val_loss",patience=15,restore_best_weights=True),
                     ReduceLROnPlateau("val_loss",factor=0.5,patience=6,min_lr=1e-6)])
    return m.predict(c, verbose=0).argmax(1)

def train_svm(Xtr, ytr, Xte, seed):
    # plain linear SVM over TF-IDF 6-mers, same settings as the main study — used only to generate
    # matched per-sequence predictions for the McNemar re-test, not as a result in its own right
    clf = LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)
    clf.fit(Xtr, ytr)
    return clf.predict(Xte)

## Section A — Fragment-length sweep (matches Table 4)

7 lengths x 3 seeds. TF-IDF 6-mer encoding, re-fit on the fragmented training fold each time (no
leakage). Also fits a matched linear SVM on the same split/fragments and saves both models'
per-sequence predictions, so the SVM-vs-ResNet-18 McNemar test (Table 5 / Table S9) can be redone
directly from this notebook's output.

In [ ]:
A_RAW  = f"{OUT}/resnet18_A_fragmentation_raw.csv"
A_PRED = f"{OUT}/resnet18_A_fragmentation_predictions.csv"

A_rows = pd.read_csv(A_RAW).to_dict("records") if os.path.exists(A_RAW) else []
A_done = {(r["length"], r["seed"]) for r in A_rows}
A_pred_rows = pd.read_csv(A_PRED).to_dict("records") if os.path.exists(A_PRED) else []
print(f"[A] resuming: {len(A_done)} (length, seed) combinations already done.")

In [ ]:
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = grouped_split(X_all, y_all, grp_all, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))

    for L in LENGTHS_FULL:
        label = "full" if L is None else str(L)
        if (label, seed) in A_done:
            print(f"[A][skip] length={label} seed={seed} already done"); continue

        rng = np.random.default_rng(seed*100 + (L or 9999))
        Ftr = fragment(X_all[tr], L, rng); Fva = fragment(X_all[va], L, rng); Fte = fragment(X_all[te], L, rng)

        tfidf = make_tfidf(Ftr)
        Ttr, Tva, Tte = tfidf(Ftr), tfidf(Fva), tfidf(Fte)

        pred_resnet = train_resnet(Ttr, Tva, Tte, ytr, yva, cw)
        pred_svm    = train_svm(Ttr, ytr, Tte, seed)

        A_rows.append({"model":"resnet18_nonoise","length":label,"length_num":(L or 3000),"seed":seed,
                        **score(yte, pred_resnet)})
        A_pred_rows.extend([{"length":label,"length_num":(L or 3000),"seed":seed,"test_idx":int(i),
                              "y_true":int(t),"y_pred_resnet18":int(pr),"y_pred_svm":int(ps)}
                             for i,t,pr,ps in zip(te, yte, pred_resnet, pred_svm)])

        pd.DataFrame(A_rows).to_csv(A_RAW, index=False)
        pd.DataFrame(A_pred_rows).to_csv(A_PRED, index=False)
        print(f"[A][seed {seed}] L={label:>4}  resnet18_nonoise f1={A_rows[-1]['f1']}  bal_acc={A_rows[-1]['bal_acc']}")

A_summ = (pd.DataFrame(A_rows).groupby(["length","length_num"])[["f1","mcc","bal_acc","acc"]]
            .agg(["mean","std"]).round(4))
A_summ.columns = [f"{m}_{s}" for m,s in A_summ.columns]
A_summ = A_summ.reset_index().sort_values("length_num")
A_summ.to_csv(f"{OUT}/resnet18_A_fragmentation_summary.csv", index=False)
print(A_summ.to_string(index=False))

## Section B — Nine-seed robustness check (matches Supplementary Table S11)

300/400/500 bp, seeds 0-8 (seeds 0-2 are just Section A's results at these three lengths; seeds
3-8 are new runs here).

In [ ]:
B_RAW = f"{OUT}/resnet18_B_nineseed_raw.csv"

# seed in SEEDS: reuse what Section A already computed
B_rows = [r for r in A_rows if r["length"] in [str(l) for l in LENGTHS_ROBUST]]
B_done = {(r["length"], r["seed"]) for r in B_rows}
if os.path.exists(B_RAW):
    extra_prev = pd.read_csv(B_RAW).to_dict("records")
    for r in extra_prev:
        if (r["length"], r["seed"]) not in B_done:
            B_rows.append(r); B_done.add((r["length"], r["seed"]))
print(f"[B] resuming: {len(B_done)} (length, seed) combinations already done (including reused Section A seeds 0-2).")

In [ ]:
for seed in EXTRA_SEEDS:
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = grouped_split(X_all, y_all, grp_all, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))

    for L in LENGTHS_ROBUST:
        label = str(L)
        if (label, seed) in B_done:
            print(f"[B][skip] length={label} seed={seed} already done"); continue

        rng = np.random.default_rng(seed*100 + L)
        Ftr = fragment(X_all[tr], L, rng); Fva = fragment(X_all[va], L, rng); Fte = fragment(X_all[te], L, rng)
        tfidf = make_tfidf(Ftr)
        Ttr, Tva, Tte = tfidf(Ftr), tfidf(Fva), tfidf(Fte)

        pred = train_resnet(Ttr, Tva, Tte, ytr, yva, cw)
        B_rows.append({"model":"resnet18_nonoise","length":label,"length_num":L,"seed":seed, **score(yte, pred)})

        pd.DataFrame(B_rows).to_csv(B_RAW, index=False)
        print(f"[B][seed {seed}] L={label:>4}  f1={B_rows[-1]['f1']}  bal_acc={B_rows[-1]['bal_acc']}")

B_summ = (pd.DataFrame(B_rows).groupby(["length","length_num"])[["f1","bal_acc"]]
            .agg(["mean","std","count"]).round(4))
B_summ.columns = [f"{m}_{s}" for m,s in B_summ.columns]
B_summ = B_summ.reset_index().sort_values("length_num")
B_summ.to_csv(f"{OUT}/resnet18_B_nineseed_summary.csv", index=False)
print(B_summ.to_string(index=False))

## Section C — Encoding comparison (matches Table 7 / Supplementary Table S5)

Full-length sequences, 4 encodings (raw 6-mer, TF-IDF 6-mer, Word2Vec 6-mer, Word2Vec-TF-IDF
hybrid) x 3 seeds. The split is the same `grouped_split` used everywhere else in this notebook;
each encoder is re-fit on the (unfragmented, full-length) training fold for each seed.

In [ ]:
C_RAW = f"{OUT}/resnet18_C_encoding_raw.csv"
C_rows = pd.read_csv(C_RAW).to_dict("records") if os.path.exists(C_RAW) else []
C_done = {(r["encoding"], r["seed"]) for r in C_rows}
ENCODINGS = ["raw_6mer", "tfidf_6mer", "word2vec_6mer", "word2vec_tfidf_hybrid"]
print(f"[C] resuming: {len(C_done)} (encoding, seed) combinations already done.")

In [ ]:
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = grouped_split(X_all, y_all, grp_all, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
    Xtr, Xva, Xte = X_all[tr], X_all[va], X_all[te]   # full length, no fragmentation

    encoders = {
        "raw_6mer": make_kfreq(Xtr),
        "tfidf_6mer": make_tfidf(Xtr),
        "word2vec_6mer": make_w2v(Xtr),
        "word2vec_tfidf_hybrid": make_w2v_tfidf_hybrid(Xtr),
    }
    for enc_name in ENCODINGS:
        if (enc_name, seed) in C_done:
            print(f"[C][skip] encoding={enc_name} seed={seed} already done"); continue
        enc = encoders[enc_name]
        Etr, Eva, Ete = enc(Xtr), enc(Xva), enc(Xte)
        pred = train_resnet(Etr, Eva, Ete, ytr, yva, cw)
        C_rows.append({"model":"resnet18_nonoise","encoding":enc_name,"seed":seed, **score(yte, pred)})
        pd.DataFrame(C_rows).to_csv(C_RAW, index=False)
        print(f"[C][seed {seed}] encoding={enc_name:<22} f1={C_rows[-1]['f1']}  bal_acc={C_rows[-1]['bal_acc']}")

C_summ = (pd.DataFrame(C_rows).groupby(["encoding"])[["f1","bal_acc"]]
            .agg(["mean","std"]).round(4))
C_summ.columns = [f"{m}_{s}" for m,s in C_summ.columns]
C_summ = C_summ.reset_index()
C_summ.to_csv(f"{OUT}/resnet18_C_encoding_summary.csv", index=False)
print(C_summ.to_string(index=False))

## Section D — PR+RT amplicon (matches Table 6)

The exact protease-plus-reverse-transcriptase clinical amplicon, built from the pre-extracted
HXB2 coordinates (`t11_hxb2_coords.csv`, same file used throughout this project). Four inputs:
protease alone, RT alone, the combined PR+RT amplicon, and that amplicon with a simulated 200 bp
internal dropout. TF-IDF 6-mer encoding, re-fit per seed. A matched linear SVM is trained on the
same split/input and both models' per-sequence predictions are saved, so the SVM-vs-ResNet-18
McNemar test on the amplicon (part of Table S9/S13) can be redone directly from this output.

In [ ]:
good["pr_only"] = good["pr"]
good["rt_only"] = good["rt"]
good["pr_rt_combined"] = good["pr"] + good["rt"]

le_d = LabelEncoder(); y_good = le_d.fit_transform(good["subtype"].values)
n_cls_d = len(le_d.classes_)

# Map each PR/RT row back to its CD-HIT-97% cluster id from the full-sequence clustering computed in
# Section 3b -- same verified mapper used in T20B/T21/T22. Tries, in order: an explicit index column,
# a full-sequence text match, then a verified positional alignment (only if row counts match the full
# dataset AND the subtype columns agree row-by-row) -- never guesses silently if none of these verify.
def build_amplicon_groups(coords_df, good, df, X_all, groups_all):
    n_full = len(df)
    for col in ["orig_index", "orig_idx", "index", "row_index", "source_index", "full_index"]:
        if col in good.columns:
            try:
                idx = good[col].astype(int).values
                if idx.min() >= 0 and idx.max() < n_full:
                    return groups_all[idx], f"used explicit \'{col}\' column"
            except Exception:
                pass
    for col in ["sequence", "full_sequence", "seq", "pol_sequence", "orig_sequence"]:
        if col in good.columns:
            idx_map = {s: i for i, s in enumerate(X_all)}
            mapped = good[col].map(idx_map)
            if mapped.notna().all():
                return groups_all[mapped.astype(int).values], f"matched on \'{col}\' full-sequence text"
    if len(coords_df) == n_full and len(good) == n_full:
        if "subtype" in good.columns and "subtype" in df.columns:
            match_rate = (good["subtype"].values == df["subtype"].values).mean()
            if match_rate > 0.99:
                return groups_all[good.index.values], f"positional alignment, verified via subtype match ({match_rate:.1%})"
            else:
                print(f"Positional alignment REJECTED: subtype columns only agree {match_rate:.1%} row-by-row.")
        else:
            print("Same row count as the full dataset, but no \'subtype\' column in one or both files to "
                  "verify row order -- not risking an unverified positional mapping.")
    print(f"Could not verify a mapping back to CD-HIT-97% clusters. Columns available in "
          f"t11_hxb2_coords.csv: {list(coords_df.columns)}. Fix the mapping explicitly instead of guessing.")
    return None, "no verified mapping"

grp_good, _mapping_method = build_amplicon_groups(coords_df, good, df, X_all, grp_all)
assert grp_good is not None, "Stopping here -- see the message above."
print(f"[Section D] amplicon cluster mapping: {_mapping_method}")

INPUTS = ["pr_only", "rt_only", "pr_rt_combined", "pr_rt_dropout"]

D_RAW  = f"{OUT}/resnet18_D_amplicon_raw.csv"
D_PRED = f"{OUT}/resnet18_D_amplicon_predictions.csv"
D_rows = pd.read_csv(D_RAW).to_dict("records") if os.path.exists(D_RAW) else []
D_done = {(r["input"], r["seed"]) for r in D_rows}
D_pred_rows = pd.read_csv(D_PRED).to_dict("records") if os.path.exists(D_PRED) else []
print(f"[D] resuming: {len(D_done)} (input, seed) combinations already done.")

In [ ]:
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    dropout_rng = np.random.default_rng(seed)
    good["pr_rt_dropout"] = good["pr_rt_combined"].apply(lambda s: dropout_200nt(s, dropout_rng))

    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(good.index.values, y_good, grp_good))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(trv, y_good[trv], grp_good[trv]))
    tr, va = trv[tr_rel], trv[va_rel]
    ytr, yva, yte = y_good[tr], y_good[va], y_good[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))

    for col in INPUTS:
        if (col, seed) in D_done:
            print(f"[D][skip] input={col} seed={seed} already done"); continue
        seqs = good[col].values
        tfidf = make_tfidf(seqs[tr])
        Ttr, Tva, Tte = tfidf(seqs[tr]), tfidf(seqs[va]), tfidf(seqs[te])

        pred_resnet = train_resnet(Ttr, Tva, Tte, ytr, yva, cw)
        pred_svm    = train_svm(Ttr, ytr, Tte, seed)

        D_rows.append({"model":"resnet18_nonoise","input":col,"seed":seed, **score(yte, pred_resnet)})
        D_pred_rows.extend([{"input":col,"seed":seed,"test_idx":int(i),"y_true":int(t),
                              "y_pred_resnet18":int(pr),"y_pred_svm":int(ps)}
                             for i,t,pr,ps in zip(te, yte, pred_resnet, pred_svm)])

        pd.DataFrame(D_rows).to_csv(D_RAW, index=False)
        pd.DataFrame(D_pred_rows).to_csv(D_PRED, index=False)
        print(f"[D][seed {seed}] input={col:<16} f1={D_rows[-1]['f1']}  bal_acc={D_rows[-1]['bal_acc']}")

D_summ = (pd.DataFrame(D_rows).groupby(["input"])[["f1","bal_acc"]]
            .agg(["mean","std"]).round(4))
D_summ.columns = [f"{m}_{s}" for m,s in D_summ.columns]
D_summ = D_summ.reset_index()
D_summ.to_csv(f"{OUT}/resnet18_D_amplicon_summary.csv", index=False)
print(D_summ.to_string(index=False))

## Section E — Ambiguity injection (matches the all-models ambiguity table)

3 lengths (300, 500, full) x 3 ambiguity rates (0%, 5%, 10%) x 3 seeds. Ambiguity (IUPAC codes)
is injected on the **test** sequences only, after fragmentation — the model is trained on clean
data, matching how a deployed classifier would actually be trained once and used on real,
possibly-ambiguous samples. TF-IDF is fit on the clean, fragmented training set only.

In [ ]:
E_RAW = f"{OUT}/resnet18_E_ambiguity_raw.csv"
E_rows = pd.read_csv(E_RAW).to_dict("records") if os.path.exists(E_RAW) else []
E_done = {(r["length"], r["ambiguity_rate"], r["seed"]) for r in E_rows}
print(f"[E] resuming: {len(E_done)} (length, rate, seed) combinations already done.")

In [ ]:
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = grouped_split(X_all, y_all, grp_all, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
    rng = np.random.default_rng(seed)

    for L in LENGTHS_AMBIG:
        label = "full" if L is None else str(L)
        Ftr = fragment(X_all[tr], L, rng); Fva = fragment(X_all[va], L, rng)
        tfidf = make_tfidf(Ftr)
        Ttr, Tva = tfidf(Ftr), tfidf(Fva)

        for rate in AMBIG_RATES:
            if (label, rate, seed) in E_done:
                print(f"[E][skip] length={label} rate={rate} seed={seed} already done"); continue
            Fte_clean = fragment(X_all[te], L, rng)
            Fte = [inject_ambiguity(s, rate, rng) for s in Fte_clean]
            Tte = tfidf(Fte)

            pred = train_resnet(Ttr, Tva, Tte, ytr, yva, cw)
            E_rows.append({"model":"resnet18_nonoise","length":label,"ambiguity_rate":rate,"seed":seed,
                            **score(yte, pred)})
            pd.DataFrame(E_rows).to_csv(E_RAW, index=False)
            print(f"[E][seed {seed}] L={label:>4} rate={rate:.2f}  f1={E_rows[-1]['f1']}  bal_acc={E_rows[-1]['bal_acc']}")

E_summ = (pd.DataFrame(E_rows).groupby(["length","ambiguity_rate"])[["f1","bal_acc"]]
            .agg(["mean","std"]).round(4))
E_summ.columns = [f"{m}_{s}" for m,s in E_summ.columns]
E_summ = E_summ.reset_index()
E_summ.to_csv(f"{OUT}/resnet18_E_ambiguity_summary.csv", index=False)
print(E_summ.to_string(index=False))

### 11. Zip all outputs for download

In [ ]:
!cd /kaggle/working && zip -r resnet18_full_reverification_outputs.zip \
    resnet18_A_fragmentation_raw.csv resnet18_A_fragmentation_predictions.csv resnet18_A_fragmentation_summary.csv \
    resnet18_B_nineseed_raw.csv resnet18_B_nineseed_summary.csv \
    resnet18_C_encoding_raw.csv resnet18_C_encoding_summary.csv \
    resnet18_D_amplicon_raw.csv resnet18_D_amplicon_predictions.csv resnet18_D_amplicon_summary.csv \
    resnet18_E_ambiguity_raw.csv resnet18_E_ambiguity_summary.csv